In [1]:
!pip install --upgrade transformers==4.45.0 huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 88.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0


In [2]:
import os
os.environ["WANDB_DISABLED"] = "true"
import re
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings
warnings.filterwarnings("ignore")

2026-03-08 11:08:04.849891: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772968085.035920      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772968085.091074      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772968085.520860      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772968085.520894      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772968085.520897      55 computation_placer.cc:177] computation placer alr

We suggest using a single class, it will make refinement easier. 

In your implementation, feel free to update the training procedure, change model and do whatever feels right 

In [10]:
# =============================================================================
#  Hand-crafted code features  (8 language-agnostic stylometric signals)
# =============================================================================

NUM_CODE_FEATURES = 8

_KEYWORDS = frozenset({
    'if', 'else', 'elif', 'for', 'while', 'return', 'def', 'class',
    'import', 'from', 'in', 'not', 'and', 'or', 'True', 'False', 'None',
    'try', 'except', 'finally', 'with', 'as', 'is', 'lambda', 'pass',
    'break', 'continue', 'raise', 'yield', 'del', 'global', 'nonlocal',
    'assert', 'async', 'await',
    'int', 'str', 'float', 'bool', 'list', 'dict', 'set', 'tuple',
    'print', 'range', 'len', 'self', 'super', 'type', 'isinstance',
    'var', 'let', 'const', 'function', 'public', 'private', 'protected',
    'static', 'void', 'new', 'this', 'null', 'undefined',
})


def extract_code_features(code: str) -> list:
    """
    Extract 8 language-agnostic stylometric features (all ≈ [0, 1]).

      1.  comment_ratio          – fraction of lines that are comments
      2.  avg_line_length        – mean chars per non-empty line (normalised)
      3.  line_length_cv         – coefficient of variation of line lengths
      4.  whitespace_consistency – std-dev of leading whitespace (normalised)
      5.  max_nesting_depth      – deepest indentation level (normalised)
      6.  blank_line_ratio       – fraction of blank lines
      7.  char_entropy           – Shannon entropy of character distribution
      8.  bracket_density        – brackets per character (normalised)
    """
    lines = code.split('\n')
    non_empty = [l for l in lines if l.strip()]
    total_lines = max(len(lines), 1)
    ne_count = max(len(non_empty), 1)

    # 1. comment_ratio
    comment_lines = sum(
        1 for l in lines
        if l.strip().startswith('#') or l.strip().startswith('//')
    )
    comment_ratio = comment_lines / total_lines

    # 2. avg_line_length
    lengths = [len(l) for l in non_empty]
    mean_len = sum(lengths) / ne_count if lengths else 0.0
    avg_line_len = min(mean_len / 200.0, 1.0)

    # 3. line_length_cv
    if lengths and mean_len > 0:
        var = sum((x - mean_len) ** 2 for x in lengths) / ne_count
        cv = math.sqrt(var) / mean_len
    else:
        cv = 0.0
    line_len_cv = min(cv / 2.0, 1.0)

    # 4. whitespace_consistency
    leading = [len(l) - len(l.lstrip()) for l in non_empty]
    if leading:
        m = sum(leading) / len(leading)
        ws_var = sum((x - m) ** 2 for x in leading) / len(leading)
        ws_std = math.sqrt(ws_var)
    else:
        ws_std = 0.0
    ws_consistency = min(ws_std / 20.0, 1.0)

    # 5. max_nesting_depth
    max_indent = max(leading) if leading else 0
    max_depth = min(max_indent / 32.0, 1.0)

    # 6. blank_line_ratio
    blank_lines = sum(1 for l in lines if not l.strip())
    blank_line_ratio = blank_lines / total_lines

    # 7. char_entropy  (Shannon entropy of character distribution)
    if len(code) > 0:
        char_counts = Counter(code)
        total_chars = len(code)
        entropy = -sum(
            (c / total_chars) * math.log2(c / total_chars)
            for c in char_counts.values()
        )
        char_entropy = min(entropy / 7.0, 1.0)
    else:
        char_entropy = 0.0

    # 8. bracket_density
    brackets = sum(1 for c in code if c in '()[]{}')
    bracket_density = min(brackets / max(len(code), 1) * 20.0, 1.0)

    return [
        comment_ratio, avg_line_len, line_len_cv,
        ws_consistency, max_depth,
        blank_line_ratio, char_entropy, bracket_density,
    ]


# =============================================================================
#  Simplified Features model  (CLS + normalised features → classifier)
#  Almost identical to baseline — only adds 8 feature dims to the classifier.
# =============================================================================

class SmoothedFeaturesClassificationModel(nn.Module):
    """
    CodeBERT + 8 hand-crafted stylometric features.
    Minimal architecture: CLS pooling → concat normalised features → classifier.
    Adds almost no extra parameters over the baseline to avoid overfitting.
    """

    def __init__(
        self,
        model_name: str,
        num_labels: int = 2,
        num_features: int = NUM_CODE_FEATURES,
        dropout: float = 0.1,
        **kwargs,
    ):
        super().__init__()
        self.num_labels = num_labels

        # Transformer backbone (same as baseline)
        self.transformer = RobertaModel.from_pretrained(model_name)
        self.config = self.transformer.config
        hidden_size = self.config.hidden_size                          # 768

        # Feature normalisation
        self.feat_norm = nn.LayerNorm(num_features)

        # Shared dropout + single classifier
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size + num_features, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None,
                code_features=None, **kwargs):
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = out.last_hidden_state[:, 0, :]                       # (B, H)
        cls_vec = self.dropout(cls_vec)

        if code_features is not None:
            feat = self.feat_norm(code_features.float())
            combined = torch.cat([cls_vec, feat], dim=-1)              # (B, H+8)
        else:
            combined = torch.cat(
                [cls_vec, cls_vec.new_zeros(cls_vec.size(0), self.feat_norm.normalized_shape[0])],
                dim=-1,
            )

        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)


# =============================================================================
#  Data collator that also stacks code_features
# =============================================================================

class FeaturesDataCollator:
    """Wraps DataCollatorWithPadding and additionally stacks code_features."""

    def __init__(self, tokenizer):
        self.base = DataCollatorWithPadding(tokenizer=tokenizer)

    def __call__(self, features):
        code_feats = None
        if 'code_features' in features[0]:
            code_feats = [f.pop('code_features') for f in features]

        batch = self.base(features)

        if code_feats is not None:
            batch['code_features'] = torch.tensor(code_feats, dtype=torch.float32)

        return batch


# =============================================================================
#  Unified Trainer  (baseline CodeBERT  /  CodeBERT + Features)
# =============================================================================

class CodeClassifierTrainer:
    """
    Handles:
      - CodeBERT baseline  (RobertaForSequenceClassification)
      - CodeBERT + Stylometric Features (SmoothedFeaturesClassificationModel)
    """

    def __init__(
        self,
        model_name: str = "microsoft/codebert-base",
        use_bilstm: bool = False,
        use_features: bool = False,
        use_smoothed_features: bool = False,
        max_length: int = 512,
        dropout: float = 0.1,
        label_smoothing: float = 0.0,
        lstm_hidden_size: int = 256,
        lstm_num_layers: int = 2,
    ):
        self.model_name = model_name
        self.use_smoothed_features = use_smoothed_features
        self.use_features = use_features
        self.use_bilstm = use_bilstm
        self.max_length = max_length
        self.dropout = dropout
        self.label_smoothing = label_smoothing
        self.tokenizer = None
        self.model = None
        self.num_labels = None

    @property
    def needs_features(self):
        return self.use_smoothed_features or self.use_features

    @property
    def tag(self):
        base = self.model_name.split("/")[-1]
        if self.use_smoothed_features:
            return f"{base}+SmoothedFeats"
        return base

    def load_and_prepare_data(self, sample_size=None, val_sample_size=None, random_seed=42):
        try:
            df = pd.read_parquet(
                '/kaggle/input/datasets/daniilor/semeval-2026-task13/'
                'SemEval-2026-Task13/task_a/task_a_training_set_1.parquet'
            )
            print(f"[{self.tag}] Dataset columns: {df.columns.tolist()}")
            print(f"[{self.tag}] Original dataset size: {len(df)}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)

            if sample_size is not None and sample_size < len(df):
                df = df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(sample_size * len(x) / len(df))),
                        random_state=random_seed,
                    )
                ).reset_index(drop=True)
                print(f"[{self.tag}] Sampled train size: {len(df)} (stratified, seed={random_seed})")

            self.num_labels = df['label'].nunique()
            print(f"[{self.tag}] Number of unique labels: {self.num_labels}")
            print(f"[{self.tag}] Train label distribution:\n{df['label'].value_counts().sort_index()}")

            val_df = pd.read_parquet(
                '/kaggle/input/datasets/daniilor/semeval-2026-task13/'
                'SemEval-2026-Task13/task_a/task_a_validation_set.parquet'
            )
            val_df = val_df.dropna(subset=['code', 'label'])
            val_df['label'] = val_df['label'].astype(int)

            if val_sample_size is not None and val_sample_size < len(val_df):
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(val_sample_size * len(x) / len(val_df))),
                        random_state=random_seed,
                    )
                ).reset_index(drop=True)
                print(f"[{self.tag}] Sampled val size: {len(val_df)} (stratified, seed={random_seed})")

            print(f"[{self.tag}] Val label distribution:\n{val_df['label'].value_counts().sort_index()}")
            print(f"[{self.tag}] Train: {len(df)},  Val: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"[{self.tag}] Error loading dataset: {e}")
            raise

    def initialize_model_and_tokenizer(self):
        print(f"[{self.tag}] Initialising model and tokenizer ...")
        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)

        if self.use_smoothed_features:
            self.model = SmoothedFeaturesClassificationModel(
                model_name=self.model_name,
                num_labels=self.num_labels,
                dropout=self.dropout,
            ).to('cuda')
        else:
            self.model = RobertaForSequenceClassification.from_pretrained(
                self.model_name,
                num_labels=self.num_labels,
                problem_type="single_label_classification",
            ).to('cuda')

        total = sum(p.numel() for p in self.model.parameters())
        train = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[{self.tag}] {self.num_labels} labels | params {total:,} | trainable {train:,}")

    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

    def prepare_datasets(self, train_df, val_df):
        print(f"[{self.tag}] Preparing datasets ...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset   = Dataset.from_pandas(val_df[['code', 'label']])

        if self.needs_features:
            def _extract_feats(examples):
                return {'code_features': [extract_code_features(c) for c in examples['code']]}
            train_dataset = train_dataset.map(_extract_feats, batched=True)
            val_dataset   = val_dataset.map(_extract_feats, batched=True)

        train_dataset = train_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])
        val_dataset   = val_dataset.map(self.tokenize_function,   batched=True, remove_columns=['code'])

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset   = val_dataset.rename_column('label', 'labels')
        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted'
        )
        return {'accuracy': accuracy, 'f1': f1, 'precision': precision, 'recall': recall}

    def train(self, train_dataset, val_dataset,
              output_dir="./results", num_epochs=10,
              batch_size=16, learning_rate=2e-5):
        print(f"[{self.tag}] Starting training ...")

        # Same training recipe for both baseline and features model
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=0.01,
            logging_dir=f'{output_dir}/logs',
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="linear",
            warmup_ratio=0.06,
            save_total_limit=2,
            report_to=[],
            fp16=True,
        )

        if self.needs_features:
            data_collator = FeaturesDataCollator(tokenizer=self.tokenizer)
        else:
            data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        print(f"[{self.tag}] Start training")
        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"[{self.tag}] Training completed. Model saved to {output_dir}")
        return trainer

    def evaluate_model(self, trainer, val_dataset):
        print(f"[{self.tag}] Evaluating ...")
        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids
        print(f"[{self.tag}] Classification Report:")
        print(classification_report(y_true, y_pred))
        return predictions

    def run_full_pipeline(self, output_dir=None,
                          num_epochs=10, batch_size=16, learning_rate=2e-5,
                          sample_size=10000, val_sample_size=None, random_seed=42):
        if output_dir is None:
            output_dir = f"./results_{self.tag.replace('+', '_')}"
        try:
            train_df, val_df = self.load_and_prepare_data(
                sample_size=sample_size,
                val_sample_size=val_sample_size,
                random_seed=random_seed,
            )
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
            )
            self.evaluate_model(trainer, val_dataset)
            print(f"[{self.tag}] Pipeline completed successfully!")
            return trainer
        except Exception as e:
            print(f"[{self.tag}] Error in pipeline: {e}")
            raise

print("SmoothedFeaturesClassificationModel,")
print("CodeClassifierTrainer defined.  (8 language-agnostic features)")

SmoothedFeaturesClassificationModel,
CodeClassifierTrainer defined.  (8 language-agnostic features)


In [11]:
# =============================================================================
#  Test-set evaluation  (4 categories, printed immediately)
# =============================================================================

SEEN_LANGS   = {'c++', 'cpp', 'python', 'java'}
UNSEEN_LANGS = {'go', 'php', 'c#', 'csharp', 'c', 'javascript', 'js'}
SEEN_DOMAINS   = {'algorithmic'}
UNSEEN_DOMAINS = {'research', 'production'}

def _norm(v):
    return str(v).strip().lower()


@torch.no_grad()
def evaluate_on_test(trainer_obj, parquet_path, max_length=256, batch_size=32, device=None):
    """Run inference on test parquet → return DataFrame with 'prediction' column."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = trainer_obj.tokenizer
    model = trainer_obj.model
    needs_features = getattr(trainer_obj, 'needs_features', False)
    model.to(device)
    model.eval()

    df = pd.read_parquet(parquet_path)
    df = df.dropna(subset=['code']).reset_index(drop=True)
    codes = df['code'].tolist()

    preds = []
    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch = codes[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, padding=True,
                        max_length=max_length, return_tensors="pt")

        fwd_kwargs = dict(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        )

        if needs_features:
            feats = [extract_code_features(c) for c in batch]
            fwd_kwargs['code_features'] = torch.tensor(feats, dtype=torch.float32).to(device)

        logits = model(**fwd_kwargs).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

    df['prediction'] = preds
    return df


def evaluate_by_category(df, tag="Model"):
    """Print classification metrics for the 4 evaluation settings + overall."""
    # detect columns
    lang_col = next((c for c in df.columns if c.lower() in ('language', 'lang', 'programming_language')), None)
    domain_col = next((c for c in df.columns if c.lower() in ('domain', 'task_type', 'category')), None)

    if 'label' not in df.columns:
        print(f"[{tag}] No 'label' column in test set — cannot evaluate.")
        return

    if lang_col is None or domain_col is None:
        print(f"[{tag}] Missing language/domain column (cols={df.columns.tolist()}). Overall only:")
        print(classification_report(df['label'], df['prediction']))
        return

    df['_l'] = df[lang_col].apply(_norm)
    df['_d'] = df[domain_col].apply(_norm)

    settings = [
        ("(i)   Seen Lang & Seen Domain",      SEEN_LANGS,   SEEN_DOMAINS),
        ("(ii)  Unseen Lang & Seen Domain",     UNSEEN_LANGS, SEEN_DOMAINS),
        ("(iii) Seen Lang & Unseen Domain",     SEEN_LANGS,   UNSEEN_DOMAINS),
        ("(iv)  Unseen Lang & Unseen Domain",   UNSEEN_LANGS, UNSEEN_DOMAINS),
    ]

    print(f"\n{'='*70}")
    print(f"  TEST RESULTS  --  {tag}")
    print(f"{'='*70}")

    for name, langs, domains in settings:
        mask = df['_l'].isin(langs) & df['_d'].isin(domains)
        sub = df[mask]
        n = len(sub)
        if n == 0:
            print(f"\n  {name}:  ** no samples **")
            continue
        y_t, y_p = sub['label'].values, sub['prediction'].values
        acc = accuracy_score(y_t, y_p)
        p, r, f1, _ = precision_recall_fscore_support(y_t, y_p, average='weighted', zero_division=0)
        print(f"\n  {name}  (n={n})")
        print(f"    Accuracy={acc:.4f}  Prec={p:.4f}  Recall={r:.4f}  F1={f1:.4f}")
        print(classification_report(y_t, y_p, zero_division=0))

    # overall
    acc = accuracy_score(df['label'], df['prediction'])
    _, _, f1, _ = precision_recall_fscore_support(df['label'], df['prediction'], average='weighted', zero_division=0)
    print(f"\n  OVERALL  (n={len(df)})  Accuracy={acc:.4f}  F1={f1:.4f}")
    print("="*70)

    df.drop(columns=['_l', '_d'], inplace=True)

print("evaluate_on_test(), evaluate_by_category() defined.")

evaluate_on_test(), evaluate_by_category() defined.


In [12]:
# =============================================================================
#  Run selected models, evaluate test set RIGHT AFTER each trains
# =============================================================================

def run_selected_models(
    configs,
    num_epochs=10,
    batch_size=16,
    learning_rate=2e-5,
    max_length=256,
    sample_size=2000,
    val_sample_size=500,
    random_seed=42,
    test_parquet="/kaggle/input/datasets/daniilor/semeval-2026-task13/"
                 "SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet",
    output_base="/kaggle/working",
):
    """
    Train selected model variants.  Immediately after each model trains,
    run inference on the test set and print results for all 4 categories.
    """
    for cfg in configs:
        trainer_obj = CodeClassifierTrainer(
            model_name=cfg["model_name"],
            use_bilstm=cfg.get("use_bilstm", False),
            use_features=cfg.get("use_features", False),
            use_smoothed_features=cfg.get("use_smoothed_features", False),
            max_length=max_length,
            dropout=cfg.get("dropout", 0.3),
            label_smoothing=cfg.get("label_smoothing", 0.1),
        )
        tag = trainer_obj.tag
        out_dir = os.path.join(output_base, f"results_{tag.replace('+', '_')}")

        print("\n" + "=" * 70)
        print(f"  MODEL: {tag}")
        print("=" * 70)

        hf_trainer = trainer_obj.run_full_pipeline(
            output_dir=out_dir,
            num_epochs=num_epochs,
            batch_size=batch_size,
            learning_rate=learning_rate,
            sample_size=sample_size,
            val_sample_size=val_sample_size,
            random_seed=random_seed,
        )

        # --- IMMEDIATELY evaluate on test set ---
        test_df = evaluate_on_test(
            trainer_obj=trainer_obj,
            parquet_path=test_parquet,
            max_length=max_length,
            batch_size=batch_size * 2,
            device="cuda",
        )
        evaluate_by_category(test_df, tag=tag)

        # free GPU memory before next model
        del hf_trainer, trainer_obj
        torch.cuda.empty_cache()

print("run_selected_models() defined.")

run_selected_models() defined.


In [ ]:
# =============================================================================
#  Run CodeBERT + Smoothed Features  (10 k samples)
# =============================================================================

CONFIGS = [
    {
        "id": 2,
        "label": "CodeBERT + Smoothed Features",
        "model_name": "microsoft/codebert-base",
        "use_bilstm": False,
        "use_features": False,
        "use_smoothed_features": True,
        "dropout": 0.3,
        "label_smoothing": 0.0,
    },
]

print(f"▶ Running {len(CONFIGS)} variants on 10 k stratified training samples:")
for c in CONFIGS:
    print(f"  • {c['label']}")
print()

run_selected_models(
    configs=CONFIGS,
    num_epochs=5,
    batch_size=16,
    learning_rate=2e-5,
    max_length=512,
    sample_size=10000,
    val_sample_size=2500,
    random_seed=42,
)